# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaiRagab/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/MaiRagab/flyrank_ml_internship.git

Cloning into 'flyrank_ml_internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 138 (delta 49), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 10.56 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [4]:
!ls /content/flyrank_ml_internship/data/raw

content_refresh_anonymized.csv


In [5]:
!find /content/flyrank_ml_internship -name "*.csv"

/content/flyrank_ml_internship/outputs/refresh_queue_sample.csv
/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv


In [6]:
import pandas as pd

df = pd.read_csv(
    "/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv"
)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [8]:
df[["days_since_last_update", "ctr"]].describe()

,days_since_last_update,ctr
count,30000.000000,30000.000000
mean,46.098300,0.510733
std,42.078709,3.279162
min,1.000000,0.000000
25%,20.000000,0.000000
50%,20.000000,0.070000
75%,104.000000,0.290000
max,373.000000,100.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule refreshes old content with poor click-through performance.

Reason Codes:

- STALE_LOW_CTR
- STALE
- LOW_CTR
- NO_ACTION

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df=pd.read_csv(
"/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv"
)

In [10]:
# Action Label
def get_action(row):
    if row["days_since_last_update"]>=104 and row["ctr"]<=0.07:
        return "REFRESH"
    return "NO_ACTION"


# Reason Code
def get_reason(row):
    if row["days_since_last_update"]>=104 and row["ctr"]<=0.07:
        return "STALE_LOW_CTR"
    elif row["days_since_last_update"]>=104:
        return "STALE"
    elif row["ctr"]<=0.07:
        return "LOW_CTR"
    return "NO_ACTION"
df["action"]=df.apply(get_action,axis=1)
df["reason_code"]=df.apply(get_reason,axis=1)

In [20]:
def get_score(row):
    score=0
    if row["days_since_last_update"]>104:
        score+=50
    if row["ctr"]<0.07:
        score+=50
    return score
df["score"]=df.apply(get_score,axis=1)

In [21]:
ranked_df=df.sort_values(
    by=["score",
        "days_since_last_update",
        "ctr"],
    ascending=[False,False,True]
)

In [22]:
import os
os.makedirs(
"/content/flyrank_ml_internship/work/outputs",
exist_ok=True
)
ranked_df.to_csv(
"/content/flyrank_ml_internship/work/outputs/baseline_action_score.csv",
index=False
)

In [23]:
top20=ranked_df.head(20)

top20[[
"action",
"reason_code",
"score",
"days_since_last_update",
"ctr"

]]

,action,reason_code,score,days_since_last_update,ctr
26242,REFRESH,STALE_LOW_CTR,100,373,0.0
29384,REFRESH,STALE_LOW_CTR,100,373,0.0
18440,REFRESH,STALE_LOW_CTR,100,372,0.0
24216,REFRESH,STALE_LOW_CTR,100,372,0.0
8631,REFRESH,STALE_LOW_CTR,100,334,0.0
15608,REFRESH,STALE_LOW_CTR,100,334,0.0
7509,REFRESH,STALE_LOW_CTR,100,313,0.0
15790,REFRESH,STALE_LOW_CTR,100,313,0.0
18841,REFRESH,STALE_LOW_CTR,100,313,0.0
21984,REFRESH,STALE_LOW_CTR,100,313,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? Seasonal traffic changes.

2. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? Low impressions affected CTR.

3. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? The content targets a niche audience.

4. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? Ranking changes affected CTR.

5. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? The content may still be evergreen.

6. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? User intent may have changed.

7. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? CTR may be temporarily low.

8. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? Search trends may have shifted.

9. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? The page may have limited visibility.

10. Action: REFRESH | Reason Code: STALE_LOW_CTR | Confidence: High | What would make it wrong? External factors may affect engagement.

11-20. Same observations apply: old content with very low CTR is prioritized for refresh, but factors such as seasonality, ranking changes, niche audiences, or limited exposure could make the recommendation less reliable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:
- Evergreen content may not need refreshing despite low CTR.
- Seasonal content may temporarily have low engagement.
- Low visibility in search results can affect CTR.

Leakage check:
- No product flags were used.
- No future windows or label-derived inputs were used.
- The rule uses only observed dataset signals (days_since_last_update and ctr).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [27]:
!pwd

/content


In [28]:
!ls

flyrank_ml_internship  sample_data


In [30]:
%cd /content/flyrank_ml_internship

/content/flyrank_ml_internship


In [31]:
!git add .
!git commit -m "Complete ML-07 assignment"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ac2a6a509a94.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.